In [28]:
# 1. Cài đặt các thư viện cần thiết
!pip install transformers accelerate datasets sentencepiece -q

import torch
from transformers import T5TokenizerFast, AutoModelForSeq2SeqLM
from huggingface_hub import hf_hub_download

# Kiểm tra GPU (Phải hiện Tesla T4 mới là chuẩn)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Đang sử dụng thiết bị: {torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU (Chậm)'}")

# 2. Nạp Model và Tokenizer (Cách thủ công để tránh lỗi TypeError)
model_name = "VietAI/vit5-base"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

vocab_file = hf_hub_download(repo_id=model_name, filename="spiece.model")
tokenizer = T5TokenizerFast(vocab_file=vocab_file)

print("[+] Bước 1 & 2 hoàn tất: Tokenizer và Model đã sẵn sàng!")

[*] Đang sử dụng thiết bị: Tesla T4


Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


[+] Bước 1 & 2 hoàn tất: Tokenizer và Model đã sẵn sàng!


In [29]:
from datasets import load_dataset
import os
import pandas as pd

# 1. Đọc và làm sạch dữ liệu
file_path = "train_data.csv"

if not os.path.exists(file_path):
    print(f"[!] LỖI: Không tìm thấy file {file_path}. Hãy upload file vào Colab!")
else:
    print("[*] Đang làm sạch dữ liệu...")

    df = pd.read_csv(file_path)

    # ❗ Bỏ các dòng không cần sửa
    df = df[df["source"] != df["target"]]

    # ❗ Thêm prefix để model hiểu task
    df["source"] = "Sửa lỗi: " + df["source"]

    # Lưu lại file sạch tạm thời
    clean_file = "clean_train_data.csv"
    df.to_csv(clean_file, index=False)

    print(f"[+] Đã làm sạch dữ liệu: {len(df)} mẫu")

    # 2. Load dataset từ file đã clean
    dataset = load_dataset("csv", data_files=clean_file)

    # 3. Hàm preprocess (QUAN TRỌNG)
    def preprocess_function(examples):
        inputs = examples["source"]
        targets = examples["target"]

        model_inputs = tokenizer(
            inputs,
            max_length=128,
            truncation=True,
            padding="max_length"
        )

        labels = tokenizer(
            text_target=targets,
            max_length=128,
            truncation=True,
            padding="max_length"
        )

        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    # 4. Tokenize dataset
    print("[*] Đang tokenize dữ liệu...")
    tokenized_datasets = dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=dataset["train"].column_names  # ❗ rất quan trọng
    )

    # 5. Chia train/test
    split_data = tokenized_datasets["train"].train_test_split(test_size=0.1)

    print(f"[+] Hoàn tất: {len(split_data['train'])} mẫu train | {len(split_data['test'])} mẫu test")

[*] Đang làm sạch dữ liệu...
[+] Đã làm sạch dữ liệu: 536 mẫu


Generating train split: 0 examples [00:00, ? examples/s]

[*] Đang tokenize dữ liệu...


Map:   0%|          | 0/536 [00:00<?, ? examples/s]

[+] Hoàn tất: 482 mẫu train | 54 mẫu test


In [33]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# 1. Thiết lập tham số
training_args = Seq2SeqTrainingArguments(
    output_dir="./vit5_spelling_weights",
    eval_strategy="epoch",     # ✅ dùng cái này (code cũ của bạn đúng)
    save_strategy="epoch",     # ✅ vẫn dùng được
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=10,       # ✅ giữ nguyên
    predict_with_generate=True,
    fp16=True,
    logging_steps=10,
    report_to="none"
)

# 2. Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=split_data["train"],
    eval_dataset=split_data["test"],
    processing_class=tokenizer,   # ✅ dùng cái này
    data_collator=data_collator,
)

# 4. Train
print("--- ĐANG HUẤN LUYỆN ---")
trainer.train()

# 5. Save model
trainer.save_model("./vit5_finished_model")
tokenizer.save_pretrained("./vit5_finished_model")  # ✅ thêm dòng này

print("[+] HOÀN THÀNH! Model đã lưu tại 'vit5_finished_model'")

--- ĐANG HUẤN LUYỆN ---


Epoch,Training Loss,Validation Loss
1,0.037320,0.035158
2,0.031950,0.036890
3,0.031386,0.026142
4,0.030554,0.031114
5,0.026107,0.026924
6,0.024888,0.030071
7,0.022165,0.015083
8,0.021122,0.054511
9,0.017283,0.030867
10,0.016606,0.043409


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[+] HOÀN THÀNH! Model đã lưu tại 'vit5_finished_model'


In [37]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("--- DEMO FIX TOKENIZER ---")

device = "cuda" if torch.cuda.is_available() else "cpu"

public_model = "iAmHieu2012/vit5-vietnamese-spelling-correction"

# ✅ FIX QUAN TRỌNG
tokenizer_demo = AutoTokenizer.from_pretrained(public_model)
model_demo = AutoModelForSeq2SeqLM.from_pretrained(public_model).to(device)
model_demo.eval()

def emergency_demo(text):
    inputs = tokenizer_demo(
        text,
        return_tensors="pt",
        max_length=128,
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model_demo.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=128,
            num_beams=5,
            early_stopping=True
        )

    return tokenizer_demo.decode(outputs[0], skip_special_tokens=True).strip()

# Test
test_sentences = [
    "Học sinh ddang học bài tasng lớp.",
    "Hnoay tơi ddi h0c ở truwờng PTIT.",
    "Toi rat thich lap trinh AI"
]

for t in test_sentences:
    print(f"Lỗi: {t}")
    print(f"Sửa: {emergency_demo(t)}")
    print("-" * 40)

--- DEMO FIX TOKENIZER ---


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Lỗi: Học sinh ddang học bài tasng lớp.
Sửa: Học sinh đang học bài tập từng lớp.
----------------------------------------
Lỗi: Hnoay tơi ddi h0c ở truwờng PTIT.
Sửa: Sau đó tôi đi học ở trường PTIT.
----------------------------------------
Lỗi: Toi rat thich lap trinh AI
Sửa: Toi rât thich lap trinh AI
----------------------------------------


In [40]:
def debug_my_model(text):
    input_text = "Sửa lỗi: " + text

    inputs = tokenizer_my(
        input_text,
        return_tensors="pt",
        max_length=128,
        truncation=True,
        padding=True
    ).to(device)

    with torch.no_grad():
        outputs = model_my.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=128,
            num_beams=5
        )

    print("RAW TOKENS:", outputs)
    print("DECODE:", tokenizer_my.decode(outputs[0]))

In [ ]:
!zip -r model_fix.zip ./vit5_finished_model

  adding: vit5_finished_model/ (stored 0%)
  adding: vit5_finished_model/tokenizer_config.json (deflated 84%)
  adding: vit5_finished_model/tokenizer.json (deflated 94%)
  adding: vit5_finished_model/config.json (deflated 51%)
  adding: vit5_finished_model/training_args.bin (deflated 53%)
  adding: vit5_finished_model/generation_config.json (deflated 31%)
  adding: vit5_finished_model/model.safetensors